# Sampling snacks

Ask a small question, keep the useful definition, and compose it into a Chat.

Install `chatsnack[typesafe]` and set `TYPESAFE_API_KEY` before running. The Chat composition cell also uses your OpenAI key. Calls evaluate live data; outputs are intentionally not saved here.

In [ ]:
from chatsnack import Chat, Question, Sampler

sample = Sampler(data="buttered popcorn").ask("Is this crunchy?")
print(sample.answer.yes)


## Keep the crunch

A Question is a reusable asset. The Sampler keeps its connection live.

In [ ]:
crunchy = Question(
    name="crunchy",
    question="Is this crunchy?",
    yes="It makes a crisp cracking sound when bitten.",
)
crunchy.save()

review = Sampler(name="SnackCheck", data="{snack}", questions=["{question.crunchy}"])
print(review.yaml)
review.save()


The saved definition stays small:

```yaml
data: "{snack}"
questions:
  - "{question.crunchy}"
```

In [ ]:
review = Sampler(name="SnackCheck")
review.load()
sample = review.ask(snack="popcorn")
print(sample.answer.choice, sample.answer.score)


## A few questions together

Use names once several questions enter the picture. The collection keeps the order we authored.

In [ ]:
category = Question(name="category", question="What kind of food is this?", choices=["snack", "meal"])
sweetness = Question(name="sweetness", question="How sweet is this?", levels=["Not sweet", "Very sweet"])

sample = review.ask(questions=["{question.crunchy}", category, sweetness], snack="popcorn")
print(sample.answers["category"].choice)
print(sample.answers["sweetness"].score)


## Bring it into a Chat

The decision stays in the authored prompt. Repeated result fields share one evaluation during this expansion.

In [ ]:
description = Chat(
    "Describe {snack} in one sentence. "
    "Crunchy: {sampler.SnackCheck.crunchy.choice}. "
    "Probability of yes: {sampler.SnackCheck.crunchy.score}."
)
print(description.ask(snack="popcorn"))


## Keep what was evaluated

Replay embeds the resolved inputs and reported model. Later Question edits won't change it.

In [ ]:
replay = Sampler.from_sample(sample, name="PopcornReview")
replay.save()
repeated = replay.ask()
print(repeated.answers["category"].choice)
print(repeated.model, repeated.usage)
